# 05 | Model training and cross-validation

Initial experiments and comparison of eight model configurations: four standard and four class-weighted.

**Source:** `archive/Project_fin_original.ipynb`. The original computations, structure, and results have been preserved. The notebook text has been translated into English; standalone setup and local checkpoints support reproducible execution.


In [ ]:
# Standalone setup: no src/ folder required.
# Run the notebooks in numerical order (02 -> 07); 01 is optional exploration.
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

working_dir = Path.cwd().resolve()
# Works when VS Code/Jupyter starts in the repository root or notebooks/.
if working_dir.name == "notebooks":
    project_root = working_dir.parent
elif (working_dir / "notebooks").is_dir():
    project_root = working_dir
elif (working_dir.parent / "notebooks").is_dir():
    project_root = working_dir.parent
else:
    project_root = working_dir

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

HORIZON = pd.Timedelta(days=10)
CUTOFFS = pd.date_range(start="2018-10-11", end="2018-11-10", freq="10D")
FEATURES_12 = [
    "n_sessions_10d", "n_active_days_10d", "n_songs_10d",
    "n_likes_10d", "n_dislikes_10d", "n_errors_10d",
    "n_playlist_adds_10d", "total_song_length_10d", "n_artists_10d",
    "downgraded_10d", "is_paid", "songs_trend_10d"
]
EVENT_COLS = [
    "userId", "time", "registration", "page", "sessionId", "level",
    "gender", "length", "artist", "song", "status", "method",
    "userAgent", "auth", "firstName", "lastName", "itemInSession", "location"
]

def raw_path(kind):
    if kind not in ("train", "test"):
        raise ValueError("kind must be 'train' or 'test'")
    filename = f"{kind}.parquet"
    candidates = [
        project_root / filename,
        project_root / "data" / "raw" / filename,
        project_root / "notebooks" / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {filename}. Put it in the repository root or data/raw/. "
        f"Current root: {project_root}"
    )

def load_events(kind):
    frame = pd.read_parquet(raw_path(kind), columns=EVENT_COLS)
    frame["userId"] = frame["userId"].astype(str)
    frame["time"] = pd.to_datetime(frame["time"])
    frame["registration"] = pd.to_datetime(frame["registration"])
    return frame

def data_path(filename):
    folder = project_root / "data" / "processed"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def artifact_path(filename):
    folder = project_root / "artifacts"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def save_cv_state(dev_data, holdout_data, cv_splits):
    dev_data.to_parquet(data_path("dev_data.parquet"), index=False)
    holdout_data.to_parquet(data_path("holdout_data.parquet"), index=False)
    fold_arrays = {}
    for i, (idx_train, idx_valid) in enumerate(cv_splits):
        fold_arrays[f"train_{i}"] = np.asarray(idx_train, dtype=np.int64)
        fold_arrays[f"valid_{i}"] = np.asarray(idx_valid, dtype=np.int64)
    np.savez_compressed(data_path("cv_splits.npz"), **fold_arrays)

def load_cv_state():
    dev = pd.read_parquet(data_path("dev_data.parquet"))
    holdout = pd.read_parquet(data_path("holdout_data.parquet"))
    with np.load(data_path("cv_splits.npz")) as saved:
        n = sum(name.startswith("train_") for name in saved.files)
        splits = [
            (saved[f"train_{i}"].copy(), saved[f"valid_{i}"].copy())
            for i in range(n)
        ]
    return dev, holdout, splits

print("Project folder:", project_root)

import joblib
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
dev_data, holdout_data, cv_splits = load_cv_state()


# Model Training

In [21]:
from sklearn.impute import SimpleImputer

## Define models and preprocessing pipelines
models = {
    "LogisticRegression": make_pipeline(
        SimpleImputer(
            strategy="constant",
            fill_value=0,
            add_indicator=True
        ),
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    ),

    "RandomForest": make_pipeline(
        SimpleImputer(
            strategy="constant",
            fill_value=0,
            add_indicator=True
        ),
        RandomForestClassifier(
            n_estimators=150,
            max_depth=10,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )
    ),

    "HistGradientBoosting": make_pipeline(
        SimpleImputer(
            strategy="constant",
            fill_value=0,
            add_indicator=True
        ),
        HistGradientBoostingClassifier(
            max_iter=150,
            learning_rate=0.05,
            max_leaf_nodes=15,
            early_stopping=False,
            random_state=42
        )
    ),

    "CatBoost": make_pipeline(
        SimpleImputer(
            strategy="constant",
            fill_value=0,
            add_indicator=True
        ),
        CatBoostClassifier(
            iterations=250,
            depth=5,
            learning_rate=0.05,
            loss_function="Logloss",
            verbose=False,
            random_seed=42
        )
    )
}

# 2. Cross-validation
results = []
validation_predictions = {}

for name, model in models.items():

    fold_aucs = []
    fold_aps = []

    print(f"\n--- {name} ---")

    for fold, (train_idx, valid_idx) in enumerate(cv_splits, 1):

        train_fold = dev_data.iloc[train_idx]
        valid_fold = dev_data.iloc[valid_idx]

        X_tr = train_fold[FEATURES_12]
        y_tr = train_fold["churn"]

        X_va = valid_fold[FEATURES_12]
        y_va = valid_fold["churn"]

        # Fit model
        model.fit(X_tr, y_tr)

        # Churn predictions
        proba = model.predict_proba(X_va)[:, 1]

        auc = roc_auc_score(y_va, proba)
        ap = average_precision_score(y_va, proba)

        fold_aucs.append(auc)
        fold_aps.append(ap)

        print(f"Fold {fold} : ROC-AUC = {auc:.4f} | AP = {ap:.4f}")

        # Save predictions for the final fold
        if fold == len(cv_splits):
            validation_predictions[name] = pd.DataFrame({
                "userId": valid_fold["userId"].values,
                "churn_reel": y_va.values,
                "proba_churn": proba,
                "churn_predit": (proba >= 0.5).astype(int)
            })

    results.append({
        "Model": name,
        "ROC_AUC_mean": np.mean(fold_aucs),
        "ROC_AUC_std": np.std(fold_aucs),
        "AP_mean": np.mean(fold_aps)
    })

# 3. Final model comparison
results_df = pd.DataFrame(results).sort_values(
    "ROC_AUC_mean",
    ascending=False
)

display(results_df)

# 4. First predictions from the best model
best_model_name = results_df.iloc[0]["Model"]

print("\nBest provisional model:", best_model_name)

display(
    validation_predictions[best_model_name]
    .sort_values("proba_churn", ascending=False)
    .head(10)
)



--- LogisticRegression ---


C:\Users\hugov\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['songs_trend_10d']. At least one non-missing value is needed for imputation with strategy='constant'.
  warnings.warn(
C:\Users\hugov\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['songs_trend_10d']. At least one non-missing value is needed for imputation with strategy='constant'.
  warnings.warn(


Fold 1 : ROC-AUC = 0.6580 | AP = 0.1044
Fold 2 : ROC-AUC = 0.7007 | AP = 0.1309

--- RandomForest ---


C:\Users\hugov\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['songs_trend_10d']. At least one non-missing value is needed for imputation with strategy='constant'.
  warnings.warn(
C:\Users\hugov\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['songs_trend_10d']. At least one non-missing value is needed for imputation with strategy='constant'.
  warnings.warn(


Fold 1 : ROC-AUC = 0.6370 | AP = 0.0985
Fold 2 : ROC-AUC = 0.6735 | AP = 0.1236

--- HistGradientBoosting ---


C:\Users\hugov\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['songs_trend_10d']. At least one non-missing value is needed for imputation with strategy='constant'.
  warnings.warn(
C:\Users\hugov\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['songs_trend_10d']. At least one non-missing value is needed for imputation with strategy='constant'.
  warnings.warn(


... (2 outputs left out)

Fold 1 : ROC-AUC = 0.6474 | AP = 0.0951
Fold 2 : ROC-AUC = 0.6880 | AP = 0.1318


,Model,ROC_AUC_mean,ROC_AUC_std,AP_mean
0,LogisticRegression,0.679346,0.021365,0.117634
3,CatBoost,0.667706,0.020305,0.113487
2,HistGradientBoosting,0.659957,0.016060,0.105064
1,RandomForest,0.655256,0.018217,0.111044



Best provisional model: LogisticRegression


,userId,churn_reel,proba_churn,churn_predit
157,1554108,0,0.462374,0
329,1435005,0,0.373674,0
2447,1551170,0,0.355330,0
278,1650232,1,0.303970,0
2265,1798061,0,0.293706,0
2410,1256037,0,0.285029,0
2443,1060993,1,0.276102,0
4,1240184,0,0.263328,0
1145,1002881,1,0.261056,0
332,1012815,1,0.255226,0


In [26]:

from sklearn.impute import SimpleImputer
from sklearn.base import clone

# Common preprocessing
def make_preprocessor():
    return SimpleImputer(
        strategy="constant",
        fill_value=0,
        add_indicator=True,
        keep_empty_features=True
    )

# Four standard models and four class-weighted models
models = {
    "LogisticRegression": make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        LogisticRegression(
            max_iter=2000, random_state=42
        )
    ),

    "LogisticRegression_Balanced": make_pipeline(
        make_preprocessor(),
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    ),

    "RandomForest": make_pipeline(
        make_preprocessor(),
        RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=10,
            n_jobs=-1,
            random_state=42
        )
    ),

    "RandomForest_Balanced": make_pipeline(
        make_preprocessor(),
        RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=10,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=42
        )
    ),

    "HistGradientBoosting": make_pipeline(
        make_preprocessor(),
        HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.05,
            max_leaf_nodes=15,
            random_state=42
        )
    ),

    "HistGradientBoosting_Balanced": make_pipeline(
        make_preprocessor(),
        HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.05,
            max_leaf_nodes=15,
            class_weight="balanced",
            random_state=42
        )
    ),

    "CatBoost": make_pipeline(
        make_preprocessor(),
        CatBoostClassifier(
            iterations=300,
            depth=5,
            learning_rate=0.05,
            verbose=False,
            random_seed=42
        )
    ),

    "CatBoost_Balanced": make_pipeline(
        make_preprocessor(),
        CatBoostClassifier(
            iterations=300,
            depth=5,
            learning_rate=0.05,
            auto_class_weights="Balanced",
            verbose=False,
            random_seed=42
        )
    )
}

# Cross-validation
results = []
validation_predictions = {}

for name, pipeline in models.items():

    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(
        cv_splits, start=1
    ):

        X_tr = dev_data.iloc[train_idx][FEATURES_12]
        y_tr = dev_data.iloc[train_idx]["churn"]

        X_va = dev_data.iloc[valid_idx][FEATURES_12]
        y_va = dev_data.iloc[valid_idx]["churn"]

        # Train a fresh model for each fold
        estimator = clone(pipeline)
        estimator.fit(X_tr, y_tr)

        # Churn prediction scores
        proba = estimator.predict_proba(X_va)[:, 1]

        # Evaluation
        auc = roc_auc_score(y_va, proba)
        ap = average_precision_score(y_va, proba)

        fold_scores.append((auc, ap))

        # Keep predictions from the last fold
        if fold == len(cv_splits):
            validation_predictions[name] = pd.DataFrame({
                "userId": dev_data.iloc[valid_idx]["userId"].values,
                "churn_reel": y_va.values,
                "probability": proba,
                "prediction": (proba >= 0.5).astype(int)
            })

    scores = np.array(fold_scores)

    results.append({
        "Model": name,
        "ROC_AUC": scores[:, 0].mean(),
        "ROC_AUC_std": scores[:, 0].std(),
        "Average_Precision": scores[:, 1].mean()
    })

    print(
        f"{name}: "
        f"AUC={scores[:, 0].mean():.4f} | "
        f"AP={scores[:, 1].mean():.4f}"
    )

# Model ranking
results_df = pd.DataFrame(results).sort_values(
    "ROC_AUC", ascending=False
)

display(results_df.round(4))

# First predictions from the best model
best_model_name = results_df.iloc[0]["Model"]

print("\nBest provisional model:", best_model_name)

display(
    validation_predictions[best_model_name]
    .sort_values("probability", ascending=False)
    .head(15)
)


LogisticRegression: AUC=0.6793 | AP=0.1176
LogisticRegression_Balanced: AUC=0.6781 | AP=0.1169
RandomForest: AUC=0.6537 | AP=0.1103
RandomForest_Balanced: AUC=0.6368 | AP=0.1063
HistGradientBoosting: AUC=0.6590 | AP=0.1041
HistGradientBoosting_Balanced: AUC=0.6448 | AP=0.1089
CatBoost: AUC=0.6635 | AP=0.1100
CatBoost_Balanced: AUC=0.6468 | AP=0.1092


,Model,ROC_AUC,ROC_AUC_std,Average_Precision
0,LogisticRegression,0.6793,0.0214,0.1176
1,LogisticRegression_Balanced,0.6781,0.0218,0.1169
6,CatBoost,0.6635,0.0229,0.1100
4,HistGradientBoosting,0.6590,0.0293,0.1041
2,RandomForest,0.6537,0.0195,0.1103
7,CatBoost_Balanced,0.6468,0.0305,0.1092
5,HistGradientBoosting_Balanced,0.6448,0.0367,0.1089
3,RandomForest_Balanced,0.6368,0.0286,0.1063



Best provisional model: LogisticRegression


,userId,churn_reel,probability,prediction
157,1554108,0,0.462374,0
329,1435005,0,0.373674,0
2447,1551170,0,0.355330,0
278,1650232,1,0.303970,0
2265,1798061,0,0.293706,0
2410,1256037,0,0.285029,0
2443,1060993,1,0.276102,0
4,1240184,0,0.263328,0
1145,1002881,1,0.261056,0
332,1012815,1,0.255226,0


In [ ]:
# Save model specifications for downstream notebooks
joblib.dump(models, artifact_path("model_specs.joblib"))
results_df.to_csv(artifact_path("comparaison_modeles.csv"), index=False)
print("Model specifications and results saved.")